In [3]:
from tensorflow.keras.applications.vgg16 import VGG16,preprocess_input
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [75]:
#Import Data
data = np.load("final_dataset.npz", allow_pickle=True)

X_train = data["X_train"]
y_train = data["y_train"]
X_test = data["X_test"]
y_test = data["y_test"]

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [77]:
y_train = np.array([label.strip().replace(" ", "") for label in y_train])
y_test = np.array([label.strip().replace(" ", "") for label in y_test])

In [78]:
print("Train labels:", set(y_train))
print("Test labels:", set(y_test))

print("In test but not train:")
print(set(y_test) - set(y_train))

Train labels: {np.str_('ons_jabeur'), np.str_('albert_einstein'), np.str_('George_W.Bush'), np.str_('oussama_mellouli'), np.str_('alan_turing')}
Test labels: {np.str_('ons_jabeur'), np.str_('albert_einstein'), np.str_('George_W.Bush'), np.str_('oussama_mellouli'), np.str_('alan_turing')}
In test but not train:
set()


In [79]:
#Encode Label


encoder = LabelEncoder()

encoder.fit(y_train)

y_train_enc = encoder.transform(y_train)
y_test_enc = encoder.transform(y_test)

In [80]:
for i, label in enumerate(encoder.classes_):
    print(f"{label} -> {i}")

George_W.Bush -> 0
alan_turing -> 1
albert_einstein -> 2
ons_jabeur -> 3
oussama_mellouli -> 4


In [9]:
#Import Vgg
vgg_model = VGG16(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)

In [7]:
#Create Function get_embedding
def get_embedding(model, face_pixels):
    
    face_pixels = face_pixels.astype('float32')

    # Add batch dimension
    samples = np.expand_dims(face_pixels, axis=0)

    # VGG16 preprocessing
    samples = preprocess_input(samples)

    # Extract features
    embedding = model.predict(samples, verbose=0)

    return embedding[0]

In [84]:
svmtrainX = []

for face in X_train:
    embedding = get_embedding(vgg_model, face)
    svmtrainX.append(embedding)

svmtrainX = np.asarray(svmtrainX)

In [85]:
svmtrainX.shape

(336, 512)

In [86]:
svmtestX = []

for face in X_test:
    embedding = get_embedding(vgg_model, face)
    svmtestX.append(embedding)

svmtestX = np.asarray(svmtestX)

In [87]:
svmtestX.shape

(90, 512)

In [88]:
np.savez_compressed(
    "final_embedding.npz",
    X_train=svmtrainX,
    y_train=y_train_enc,
    X_test=svmtestX,
    y_test=y_test_enc
)

In [91]:
data = np.load("final_embedding.npz")

X_train = data["X_train"]
y_train = data["y_train"]

X_test = data["X_test"]
y_test = data["y_test"]

In [92]:
print(X_train.shape)  # embeddings
print(y_train.shape)  # encoded labels

print(X_test.shape)
print(y_test.shape)

(336, 512)
(336,)
(90, 512)
(90,)


In [1]:
import cv2
image=cv2.imread(r"C:\Users\STAR-STORE\Desktop\data science bootcamp\python\project\face.jpg")

In [4]:
image=np.array(image)

In [5]:
image.shape

(224, 224, 3)

In [10]:
image_embedding=get_embedding(vgg_model,image)

In [11]:
image_embedding.shape

(512,)

In [12]:
image_embedding=np.expand_dims(image_embedding,axis=0)


In [13]:
image_embedding.shape

(1, 512)

In [14]:
type(image_embedding)

numpy.ndarray

In [16]:
np.save("embedding.npy", image_embedding)